In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import joblib
import os


In [2]:
df = pd.read_csv("../data/Medical_appointment_data.csv")

In [3]:
df["age"]=df["age"].fillna(df["age"].median())
df["specialty"]=df["specialty"].fillna("Unknown")
df["disability"]=df["disability"].fillna("Unknown")
df["place"]=df["place"].fillna("Unknown")


In [4]:
# Weather variables: impute with mean
weather_vars = ["average_temp_day", "average_rain_day", "max_temp_day", "max_rain_day", "rainy_day_before", "storm_day_before"]
for col in weather_vars:
   df[col]=df[col].fillna(df[col].mean())


In [5]:
# --- Feature Engineering ---
# Age flags
df["under_12"] = (df["age"] < 12).astype(int)
df["over_60"] = (df["age"] > 60).astype(int)


In [6]:
# Temporal features
df["appointment_date_continuous"] = pd.to_datetime(df["appointment_date_continuous"])
df["day_of_week"] = df["appointment_date_continuous"].dt.dayofweek
df["month"] = df["appointment_date_continuous"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)


In [7]:
# Target encoding
df["no_show"] = df["no_show"].map({"yes":1, "no":0})


In [9]:
# --- Define Features & Target ---
target = "no_show"
X = df.drop(columns=[target])
y = df[target]


In [10]:
# Separate categorical and numerical features
categorical_features = ["gender", "specialty", "appointment_shift", "place", "disability", "patient_needs_companion"]
numerical_features = ["age", "under_12", "over_60", "day_of_week", "month", "is_weekend"] + weather_vars



In [12]:
# --- Preprocessing Pipelines ---
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

#preprocessor = ColumnTransformer(
  #  transformers=[
    #    ("categorical", categorical_transformer, categorical_features),
      #  ("numerical", numerical_transformer, numerical_features)
    #]
#)

# Categorical Preprocessor

preprocessor_cat = ColumnTransformer(
    transformers=[
        ("categorical", categorical_transformer, categorical_features)
    ]
)

# Numerical Preprocessor

preprocessor_nume = ColumnTransformer(
    transformers=[
        ("numerical", numerical_transformer, numerical_features)
    ]
)


In [14]:
# Ensure the folder exists
os.makedirs("models", exist_ok=True)

# --- Save preprocessor for reuse ---
#joblib.dump(preprocessor, "models/preprocessor.pkl")
#print("✅ Preprocessor saved to models/preprocessor.pkl")
joblib.dump(preprocessor_cat, "models/preprocessor_cat.pkl")
print("✅ preprocessor_cat saved to models/preprocessor_cat.pkl")
joblib.dump(preprocessor_nume, "models/preprocessor_nume.pkl")
print("✅ preprocessor_nume saved to models/preprocessor_nume.pkl")

✅ preprocessor_cat saved to models/preprocessor_cat.pkl
✅ preprocessor_nume saved to models/preprocessor_nume.pkl


In [15]:
# --- Train-Test Split ---
# For classification (random split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (87674, 30)
Test shape: (21919, 30)


In [17]:
def validate_feature_lists(df, categorical_features, numerical_features):
    """
    Check if all features exist in the DataFrame.
    Prints missing and extra columns for debugging.
    """
    df_cols = set(df.columns)
    cat_set = set(categorical_features)
    num_set = set(numerical_features)

    # Missing columns
    missing_cat = cat_set - df_cols
    missing_num = num_set - df_cols

    # Extra columns (not in DataFrame but listed)
    extra_cat = df_cols - cat_set
    extra_num = df_cols - num_set

    print("🔍 Validation Report")
    if missing_cat:
        print("Missing categorical features:", missing_cat)
    if missing_num:
        print("Missing numerical features:", missing_num)
    if not missing_cat and not missing_num:
        print("✅ All listed features exist in DataFrame.")

    # Optional: show any columns in df not accounted for
    accounted = cat_set.union(num_set)
    unaccounted = df_cols - accounted
    if unaccounted:
        print("Columns in DataFrame not in feature lists:", unaccounted)

In [18]:
validate_feature_lists(X_train, categorical_features, numerical_features)

🔍 Validation Report
✅ All listed features exist in DataFrame.
Columns in DataFrame not in feature lists: {'appointment_time', 'over_60_years_old', 'heat_intensity', 'rain_intensity', 'SMS_received', 'Diabetes', 'Scholarship', 'Alcoholism', 'Handcap', 'under_12_years_old', 'Hipertension', 'appointment_date_continuous'}


In [19]:
# --- Save Preprocessed Data (Optional) ---
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)


NameError: name 'preprocessor' is not defined